# 02 Pipeline

Build a governed pipeline in five steps: **Environment → Data Contracts → Read → Transform → Write**.

## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.2.0 | Voyce | 6 Aug 2026 |

This redesigned version has local structural and public-API compatibility validation only. Run it in your configured Fabric workspace before recording a new runtime test.

# 0. Environment

Run the shared Fabric configuration and import the public APIs used by this pipeline.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    # FabricOps v0.1.0 onwards
    read_lakehouse_table,
    widget_view_catalogue,
    # FabricOps v0.2.0 onwards
    check_freshness,
    check_source_stability,
    check_dq,
    check_schema,
    check_sensitive_data,
    observe_table,
    profile_table,
    pipeline_read,
    resolve_table_id,
    widget_select_data_contract,
    pipeline_write,
)

READ_TABLE_IDS = {}
READ_DFS = {}

catalogue_widget = widget_view_catalogue(mode="explore")

# 1. Data Contracts

Select the Data Contracts to test with this pipeline. Production automatically uses activated Data Contracts.

In [ ]:
CONTRACTS = widget_select_data_contract()

## Pipeline target

Define the target and its proposed processing. Development can author changes without a selected Data Contract; selected Development contracts validate this proposal, and Production enforces the active contract.

In [ ]:
WRITE_STORE = "unified"
WRITE_SCHEMA = "demo"
WRITE_TABLE = "curated_orders"
WRITE_LOAD_STRATEGY = "overwrite"
WRITE_LOAD_STRATEGY_PARAMETERS = {}

WRITE_TABLE_ID = resolve_table_id(
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
    table_name=WRITE_TABLE,
)

# 2. Read

Copy a complete Read block and describe the governed source. FabricOps resolves the canonical `table_id` and whether the configured source uses a Lakehouse table, Warehouse table, or Warehouse query path. Engineering checks and profiling remain explicit below.

## READ 1 — Orders

In [ ]:
READ = 1
READ_STORE = "source"
READ_SCHEMA = "demo"
READ_TABLE = "orders"
READ_QUERY = None

read_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)
read_df = read_result["dataframe"]
READ_TABLE_ID = read_result["table_id"]

if read_result["has_contract"]:
    observation = observe_table(
        READ_TABLE,
        store=READ_STORE,
        schema=READ_SCHEMA,
        target_table_id=WRITE_TABLE_ID,
    )
    check_freshness(observation, raise_on_failure=True)
    check_source_stability(observation, target_table_id=WRITE_TABLE_ID)

if read_result["is_query"]:
    # Custom SQL returns derived engineering data, not the complete governed physical table.
    read_profile = profile_table(dataframe=read_df)
else:
    check_schema(READ_TABLE_ID, dataframe=read_df, raise_on_failure=True)
    check_dq(read_df, table_id=READ_TABLE_ID, raise_on_failure=True)
    read_profile = profile_table(dataframe=read_df, table_id=READ_TABLE_ID)

display(read_profile["profile"])

READ_TABLE_IDS[READ] = READ_TABLE_ID
READ_DFS[READ] = read_df
catalogue_widget["show"](table_id=READ_TABLE_ID)

## READ 2 — Products

In [ ]:
READ = 2
READ_STORE = "source"
READ_SCHEMA = "demo"
READ_TABLE = "products"
READ_QUERY = None

read_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)
read_df = read_result["dataframe"]
READ_TABLE_ID = read_result["table_id"]

if read_result["has_contract"]:
    observation = observe_table(
        READ_TABLE,
        store=READ_STORE,
        schema=READ_SCHEMA,
        target_table_id=WRITE_TABLE_ID,
    )
    check_freshness(observation, raise_on_failure=True)
    check_source_stability(observation, target_table_id=WRITE_TABLE_ID)

if read_result["is_query"]:
    # Custom SQL returns derived engineering data, not the complete governed physical table.
    read_profile = profile_table(dataframe=read_df)
else:
    check_schema(READ_TABLE_ID, dataframe=read_df, raise_on_failure=True)
    check_dq(read_df, table_id=READ_TABLE_ID, raise_on_failure=True)
    read_profile = profile_table(dataframe=read_df, table_id=READ_TABLE_ID)

display(read_profile["profile"])

READ_TABLE_IDS[READ] = READ_TABLE_ID
READ_DFS[READ] = read_df
catalogue_widget["show"](table_id=READ_TABLE_ID)

## READ 3 — Order History

In [ ]:
READ = 3
READ_STORE = "product"
READ_SCHEMA = "demo"
READ_TABLE = "order_history"
READ_QUERY = f"""
SELECT
    customer_id,
    COUNT(*) AS historical_order_count,
    SUM(net_amount) AS historical_net_amount,
    MAX(order_datetime) AS latest_historical_order_datetime
FROM {READ_SCHEMA}.{READ_TABLE}
GROUP BY customer_id
"""

read_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)
read_df = read_result["dataframe"]
READ_TABLE_ID = read_result["table_id"]

if read_result["has_contract"]:
    observation = observe_table(
        READ_TABLE,
        store=READ_STORE,
        schema=READ_SCHEMA,
        target_table_id=WRITE_TABLE_ID,
    )
    check_freshness(observation, raise_on_failure=True)
    check_source_stability(observation, target_table_id=WRITE_TABLE_ID)

if read_result["is_query"]:
    # Custom SQL returns derived engineering data, not the complete governed physical table.
    read_profile = profile_table(dataframe=read_df)
else:
    check_schema(READ_TABLE_ID, dataframe=read_df, raise_on_failure=True)
    check_dq(read_df, table_id=READ_TABLE_ID, raise_on_failure=True)
    read_profile = profile_table(dataframe=read_df, table_id=READ_TABLE_ID)

display(read_profile["profile"])

READ_TABLE_IDS[READ] = READ_TABLE_ID
READ_DFS[READ] = read_df
catalogue_widget["show"](table_id=READ_TABLE_ID)

# 3. Transform

Use project-owned PySpark so the business transformation stays visible.

In [ ]:
orders_df = READ_DFS[1].alias("orders")
products_df = READ_DFS[2].alias("products")
history_df = READ_DFS[3].alias("history")

transformed_df = (
    orders_df
    .join(products_df, on="product_id", how="left")
    .join(history_df, on="customer_id", how="left")
    .withColumn(
        "order_net_amount",
        F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2),
    )
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)
display(transformed_df)

# 4. Write

Prepare the engineering proposal, run target Guardrails, write the treated DataFrame, and register the published target.

## WRITE 1 — Curated Orders

In [ ]:
WRITE = 1
write_df = transformed_df

check_schema(WRITE_TABLE_ID, dataframe=write_df, raise_on_failure=True)
check_dq(write_df, table_id=WRITE_TABLE_ID, raise_on_failure=True)

sensitive_result = check_sensitive_data(write_df, table_id=WRITE_TABLE_ID)
if not sensitive_result["can_continue"]:
    raise RuntimeError("A blocking Sensitive Data Guardrail failed.")
prepared_df = sensitive_result["dataframe"].persist()

write_result = pipeline_write(
    prepared_df,
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
    table_name=WRITE_TABLE,
    load_strategy=WRITE_LOAD_STRATEGY,
    load_strategy_parameters=WRITE_LOAD_STRATEGY_PARAMETERS,
    source_table_ids=[READ_TABLE_IDS[1], READ_TABLE_IDS[2], READ_TABLE_IDS[3]],
)
WRITE_TABLE_ID = write_result["table_id"]
prepared_df.unpersist()

published_df = read_lakehouse_table(
    WRITE_TABLE,
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
)
write_profile = profile_table(dataframe=published_df, table_id=WRITE_TABLE_ID)
display(write_profile["profile"])

catalogue_widget["show"](table_id=WRITE_TABLE_ID)
